In [ ]:
from langgraph.graph import StateGraph , START , END
from langchain_openai import ChatOpenAI 
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

In [ ]:
class BlogState(TypedDict):
    topic: str
    outline: str
    blog: str

In [ ]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']

    prompt = f'Generate a detailed outline for a blog post with the title: {title}'

    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline} '

    content  = model.invoke(prompt).content

    state['blog'] = content

    return state


In [ ]:
graph = StateGraph(BlogState)

graph.add_node('Create_Outline', create_outline)
graph.add_node('Create_Blog', create_blog)

graph.add_edge(START, 'Create_Outline')
graph.add_edge('Create_Outline', 'Create_Blog')
graph.add_edge('Create_Blog', END)

workflow = graph.compile()

In [ ]:
initial_state = {'title' : 'The impact of Artificial Intelligence on the job market'}

final_state = workflow.invoke(initial_state)

print(final_state['outline'])
print(final_state['content'])